# Chapter 10: the stage table and the closure gap

Section 10.5 builds the stage table from vessel events and prints the closure gap per month. The year is held
as vessel event volumes summed by month and stage (data/vessel_event_month.csv). March 2026 is the book's
typical month.

in = out + known loss + change in stock + unexplained

In [ ]:
import sys
sys.path.insert(0, "..")
import matplotlib.pyplot as plt
import pandas as pd
import kestrel as k

months = k.month_balance()
print(months[["month", "cast_hl", "known_loss_hl", "dispatched_hl", "change_in_stock_hl", "closure_gap_hl",
              "closure_gap_pct"]].to_string(index=False))

The typical month, stage by stage.

In [ ]:
stages = k.stage_table("2026-03")
print(stages.to_string(index=False))
m = months.set_index("month").loc["2026-03"]
for name, col in [("Cellar", "cellar_loss_hl"), ("Filtration and bright beer tanks", "filtration_loss_hl"),
                  ("Packaging", "packaging_loss_hl"), ("Unexplained", "closure_gap_hl")]:
    print(f"{name:34s} {100 * m[col] / m.cast_hl:5.2f} percent  {m[col]:6.0f} hL")
total = m.known_loss_hl + m.closure_gap_hl
print(f"{'Total':34s} {100 * total / m.cast_hl:5.2f} percent  {total:6.0f} hL")

In [ ]:
steps = [("cast", m.cast_hl), ("cellar", -m.cellar_loss_hl), ("filtration", -m.filtration_loss_hl),
         ("packaging", -m.packaging_loss_hl), ("unexplained", -m.closure_gap_hl)]
level, fig = m.cast_hl, plt.figure(figsize=(7, 3.5))
for i, (name, v) in enumerate(steps):
    bottom = 0 if i == 0 else level + v
    plt.bar(name, abs(v) if i else v, bottom=bottom, color="tab:red" if name == "unexplained" else "tab:blue")
    level = v if i == 0 else level + v
plt.bar("dispatched", m.dispatched_hl, color="tab:green")
plt.ylim(8200, 8750)
plt.ylabel("hL")
plt.title("March 2026, synthetic")
plt.show()

Months whose gap passes 0.5 percent of cast fail the balance check (chapter 12). Check the instruments first.

In [ ]:
print(months[months.closure_gap_pct >= 0.5][["month", "closure_gap_hl", "closure_gap_pct"]].to_string(index=False))

Extract efficiency for brew B26-0068 from the model, not from a cell (section 10.3).

In [ ]:
con = k.connect()
theory = k.theoretical_extract(con)["B26-0068"]
plato = k.gravity_of_record(con).set_index("brew_id").plato["B26-0068"]
per_hl = k.WATER_KG_PER_L * plato * k.sg_from_plato(plato)
print(f"theoretical {theory:,.0f} kg; extract per hL {per_hl:.2f} kg; cast {300 * per_hl:,.0f} kg; "
      f"efficiency {100 * 300 * per_hl / theory:.1f} percent")

Overhead per tank day (section 10.4). Fixed overhead of USD 176,000 a month is a Kestrel planning figure.

In [ ]:
OVERHEAD_USD_MONTH = 176000
DISPATCHED_PER_FILL = 574.2
tank_days_month = months.tank_days_used.sum() / 12
per_day = OVERHEAD_USD_MONTH / tank_days_month
print(f"{months.tank_days_used.sum():,} tank days a year, {tank_days_month:.0f} a month, {per_day:.0f} per tank day")
for beer, days in [("Lager", 30), ("Pale ale", 16), ("Stout", 23)]:
    print(f"{beer:9s} {days:3d} days  {days * per_day:8,.0f}  {days * per_day / DISPATCHED_PER_FILL:5.1f} per hL")